In [2]:
import pandas as pd
import numpy as np

# ---------- Load ----------
path = "marketing_campaign.csv"
df = pd.read_csv(path, sep="\t")   # IMPORTANT: file is tab-separated

# ---------- Feature engineering ----------
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], errors="coerce", dayfirst=True)
df["Age"] = pd.Timestamp("today").year - pd.to_numeric(df["Year_Birth"], errors="coerce")

df["Children"] = (
    pd.to_numeric(df["Kidhome"], errors="coerce").fillna(0)
    + pd.to_numeric(df["Teenhome"], errors="coerce").fillna(0)
)

mnt_cols = [c for c in df.columns if c.startswith("Mnt")]
df["TotalSpend"] = df[mnt_cols].sum(axis=1)

purchase_cols = [c for c in df.columns if c.startswith("Num") and c.endswith("Purchases")]
df["TotalPurchases"] = df[purchase_cols].sum(axis=1)

df["ResponseLabel"] = np.where(df["Response"].fillna(0).astype(int) == 1, "Responded", "Did not respond")

print(df.shape)
print(df.head())


(2240, 34)
     ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  Dt_Customer  Recency  MntWines  ...  AcceptedCmp2  Complain  Z_CostContact  \
0  2012-09-04       58       635  ...             0         0              3   
1  2014-03-08       38        11  ...             0         0              3   
2  2013-08-21       26       426  ...             0         0              3   
3  2014-02-10       26        11  ...             0         0              3   
4  2014-01-19       94       173  ...             0         0              3   

   Z_Revenue  Response  Age  Children  

In [9]:
import plotly.express as px
import plotly.graph_objects as go

# 1) Violin: TotalSpend by Education, split by response
plot_df1 = df.dropna(subset=["TotalSpend", "Education"]).copy()
edu_order = plot_df1.groupby("Education")["TotalSpend"].median().sort_values(ascending=False).index.tolist()

fig1 = px.violin(
    plot_df1, x="Education", y="TotalSpend",
    color="ResponseLabel",
    category_orders={"Education": edu_order},
    box=True, points="all",
    hover_data=["Income", "Age", "Children", "Recency"],
    title="Insight 1 — Spending distribution by Education (Responded vs Did not respond)"
)
fig1.write_html("viz1_violin_spend_by_education.html", include_plotlyjs="cdn")


# 2) Joint-style: Income vs TotalSpend, marginals + trendline
plot_df2 = df.dropna(subset=["Income", "TotalSpend"]).copy()
q_low, q_hi = plot_df2["Income"].quantile([0.01, 0.99])
plot_df2 = plot_df2[(plot_df2["Income"] >= q_low) & (plot_df2["Income"] <= q_hi)]

fig2 = px.scatter(
    plot_df2, x="Income", y="TotalSpend",
    color="ResponseLabel", opacity=0.65,
    marginal_x="histogram", marginal_y="histogram",
    trendline="ols",
    hover_data=["Education", "Marital_Status", "Age", "Children", "Recency", "TotalPurchases"],
    title="Insight 2 — Income vs TotalSpend with marginal distributions (and trendline)"
)
fig2.write_html("viz2_joint_income_totalspend.html", include_plotlyjs="cdn")


# 3) Density contour: Web visits vs Web purchases + responder overlay
plot_df3 = df.dropna(subset=["NumWebVisitsMonth", "NumWebPurchases"]).copy()

fig3 = go.Figure()
fig3.add_trace(go.Histogram2dContour(
    x=plot_df3["NumWebVisitsMonth"],
    y=plot_df3["NumWebPurchases"],
    contours=dict(showlabels=True),
    name="Density (all)",
    showscale=False,
    opacity=0.8
))
resp_pts = plot_df3[plot_df3["ResponseLabel"] == "Responded"]
fig3.add_trace(go.Scatter(
    x=resp_pts["NumWebVisitsMonth"],
    y=resp_pts["NumWebPurchases"],
    mode="markers",
    name="Responded (points)",
    opacity=0.7
))
fig3.update_layout(
    title="Insight 3 — Digital funnel friction: Web visits vs Web purchases",
    xaxis_title="NumWebVisitsMonth",
    yaxis_title="NumWebPurchases"
)
fig3.write_html("viz3_density_web_visits_purchases.html", include_plotlyjs="cdn")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1) Violin + box: TotalSpend by Education, hue=ResponseLabel
plot_df1 = df.dropna(subset=["TotalSpend", "Education"]).copy()
edu_order = plot_df1.groupby("Education")["TotalSpend"].median().sort_values(ascending=False).index.tolist()

plt.figure(figsize=(12, 5))
sns.violinplot(data=plot_df1, x="Education", y="TotalSpend", hue="ResponseLabel", order=edu_order, cut=0, inner="box")
plt.xticks(rotation=20)
plt.title("Insight 1 — TotalSpend distribution by Education (Responded vs Did not respond)")
plt.tight_layout()
plt.show()

# 2) Joint plot: Income vs TotalSpend (trim 1–99% to reduce outlier distortion)
plot_df2 = df.dropna(subset=["Income", "TotalSpend"]).copy()
q_low, q_hi = plot_df2["Income"].quantile([0.01, 0.99])
plot_df2 = plot_df2[(plot_df2["Income"] >= q_low) & (plot_df2["Income"] <= q_hi)]

g = sns.jointplot(
    data=plot_df2, x="Income", y="TotalSpend",
    hue="ResponseLabel",
    kind="scatter", alpha=0.5, height=7
)
g.fig.suptitle("Insight 2 — Income vs TotalSpend (Jointplot with marginals)", y=1.02)
plt.show()

# 3) 2D density: Web visits vs Web purchases + responders overlay
plot_df3 = df.dropna(subset=["NumWebVisitsMonth", "NumWebPurchases"]).copy()

plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=plot_df3,
    x="NumWebVisitsMonth", y="NumWebPurchases",
    fill=True, levels=12, thresh=0.05
)
sns.scatterplot(
    data=plot_df3[plot_df3["ResponseLabel"] == "Responded"],
    x="NumWebVisitsMonth", y="NumWebPurchases",
    s=20, alpha=0.6
)
plt.title("Insight 3 — Web visits vs Web purchases (density + responders)")
plt.tight_layout()
plt.show()
